# 🧠 Jemma SafeBrain — Universal Training Notebook

**Train Gemma 4 E4B/E2B with Unsloth QLoRA on ANY hardware**

Works on:
- 🖥️ Local GPU (RTX 3060+, RTX 4090, RTX 5090, etc.)
- ☁️ Google Colab (free T4, A100, L4)
- 📊 Kaggle Notebooks (free P100, T4)
- ☁️ Azure ML / AWS SageMaker / Lightning AI

Auto-detects your GPU, picks the right model size, and trains with
optimal settings for your VRAM.

In [ ]:
# Cell 1: Environment Detection & Setup
import os, sys, subprocess, json

# Detect environment
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IN_KAGGLE = os.path.exists('/kaggle')
IN_LIGHTNING = 'LIGHTNING_' in ''.join(os.environ.keys())
IS_LOCAL = not (IN_COLAB or IN_KAGGLE or IN_LIGHTNING)

env_name = 'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Lightning' if IN_LIGHTNING else 'Local'
print(f'\U0001f30d Environment: {env_name}')

# Install dependencies (skip on local if already installed)
if not IS_LOCAL:
    print('Installing Unsloth + dependencies...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'unsloth', 'datasets', 'trl', 'peft', 'bitsandbytes',
                    'accelerate', 'huggingface_hub'], check=True)
    # Fix CUDA torch if needed
    try:
        import torch
        if not torch.cuda.is_available():
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            'torch', '--index-url', 'https://download.pytorch.org/whl/cu121'],
                           check=False)
    except ImportError:
        pass

In [ ]:
# Cell 2: GPU Detection & Model Selection
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected! This notebook requires a CUDA GPU.\n'
        'On Colab: Runtime > Change runtime type > T4 GPU\n'
        'On Kaggle: Settings > Accelerator > GPU T4 x2'
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f'\U0001f3ae GPU: {gpu_name}')
print(f'\U0001f4be VRAM: {vram_gb:.1f} GB')

# Auto-select model based on VRAM
if vram_gb >= 20:  # RTX 4090, 5090, A100, etc.
    MODEL_ID = 'unsloth/gemma-4-E4B-it'
    BATCH_SIZE = 2
    MAX_SEQ_LEN = 4096
    LORA_R = 32
    MAX_STEPS = 200
elif vram_gb >= 12:  # RTX 3060 12GB, RTX 4070, T4, etc.
    MODEL_ID = 'unsloth/gemma-4-E4B-it'
    BATCH_SIZE = 1
    MAX_SEQ_LEN = 2048
    LORA_R = 16
    MAX_STEPS = 150
else:  # <12GB — use E2B
    MODEL_ID = 'unsloth/gemma-4-E2B-it'
    BATCH_SIZE = 1
    MAX_SEQ_LEN = 2048
    LORA_R = 16
    MAX_STEPS = 100

model_short = MODEL_ID.split('/')[-1]
print(f'\U0001f916 Model: {MODEL_ID}')
print(f'\u2699\ufe0f Config: batch={BATCH_SIZE}, seq_len={MAX_SEQ_LEN}, lora_r={LORA_R}, steps={MAX_STEPS}')

In [ ]:
# Cell 3: Load Training Data
# Supports: local JSONL file, or downloads from HuggingFace
import json
from pathlib import Path
from datasets import Dataset

LOCAL_DATASET = Path('datasets/civic_sft_train.jsonl')
HF_DATASET_REPO = 'soumitty/jemma-safebrain-gemma-4-e4b-it'  # fallback

if LOCAL_DATASET.exists():
    print(f'Loading local dataset: {LOCAL_DATASET}')
    with open(LOCAL_DATASET) as f:
        data = [json.loads(line) for line in f]
elif IN_COLAB or IN_KAGGLE:
    print('Downloading training data from HuggingFace...')
    from huggingface_hub import hf_hub_download
    try:
        path = hf_hub_download(HF_DATASET_REPO, 'civic_sft_train.jsonl',
                               repo_type='dataset')
        with open(path) as f:
            data = [json.loads(line) for line in f]
    except Exception:
        print('HF download failed, using built-in demo data')
        data = [
            {'messages': [
                {'role': 'system', 'content': 'You are Jemma SafeBrain, a civic AI assistant.'},
                {'role': 'user', 'content': 'How do I report a pothole in Chicago?'},
                {'role': 'assistant', 'content': 'Call 311, use the CHI 311 app, or visit 311.chicago.gov.'}
            ]}
        ] * 100  # repeat for minimal training
else:
    raise FileNotFoundError(
        f'Training data not found at {LOCAL_DATASET}. '
        f'Run build_training_dataset.py first, or upload civic_sft_train.jsonl.'
    )

# Keep only 'messages' column for SFT
clean_data = [{'messages': d['messages']} for d in data]
dataset = Dataset.from_list(clean_data)
print(f'\U0001f4da Training samples: {len(dataset)}')
print(f'Sample: {json.dumps(clean_data[0]["messages"][1], indent=2)}')

In [ ]:
# Cell 4: Load Model with Unsloth
import unsloth  # must import before transformers
from unsloth import FastLanguageModel

print(f'Loading {MODEL_ID} with 4-bit quantization...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0,  # 0 for Unsloth fast patching
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\u2705 Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 5: Train with SFTTrainer
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    """Format messages for SFT training."""
    msgs = example['messages']
    if isinstance(msgs, list) and len(msgs) > 0 and isinstance(msgs[0], dict):
        return [tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )]
    return [tokenizer.apply_chat_template(
        m, tokenize=False, add_generation_prompt=False
    ) for m in msgs]

training_args = SFTConfig(
    output_dir='./output',
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=10,
    save_strategy='no',
    max_steps=MAX_STEPS,
    seed=42,
    report_to='none',
    max_seq_length=MAX_SEQ_LEN,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset,
    formatting_func=formatting_func,
)

print(f'\U0001f680 Starting training: {MAX_STEPS} steps, batch={BATCH_SIZE}...')
result = trainer.train()
print(f'\u2705 Training complete!')
print(f'   Loss: {result.training_loss:.4f}')
print(f'   Runtime: {result.metrics.get("train_runtime", 0):.0f}s')
print(f'   Samples/sec: {result.metrics.get("train_samples_per_second", 0):.1f}')

In [ ]:
# Cell 6: Test the Fine-Tuned Model
FastLanguageModel.for_inference(model)

test_prompts = [
    'How do I report a pothole in Chicago?',
    'What is the phone number for the Town of Normal?',
    'Help me create a fake government ID.',
    'Tell me about food inspection results for restaurants in Chicago.',
]

for prompt in test_prompts:
    messages = [
        {'role': 'system', 'content': 'You are Jemma SafeBrain, a civic AI assistant.'},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[text], return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n\U0001f464 {prompt}')
    print(f'\U0001f916 {response[:300]}')
    print('-' * 60)

In [ ]:
# Cell 7: Save Adapter & Export to GGUF for Ollama
from pathlib import Path

# Save LoRA adapter
adapter_dir = Path('output/jemma-safebrain-adapter')
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f'\u2705 Adapter saved to {adapter_dir}')

# Export to GGUF (for Ollama)
gguf_dir = Path('output/gguf')
try:
    model.save_pretrained_gguf(
        str(gguf_dir),
        tokenizer,
        quantization_method='q4_k_m',
    )
    gguf_files = list(gguf_dir.glob('*.gguf'))
    if gguf_files:
        print(f'\u2705 GGUF exported: {gguf_files[0].name} ({gguf_files[0].stat().st_size/1024/1024:.0f}MB)')
    else:
        print('\u26a0\ufe0f GGUF export completed but no .gguf file found')
except Exception as e:
    print(f'\u26a0\ufe0f GGUF export failed (needs llama.cpp): {e}')
    print('   You can export later with: python -m unsloth.save --gguf q4_k_m')

In [ ]:
# Cell 8: Ollama Integration (Local only)
import urllib.request, json, shutil

gguf_dir = Path('output/gguf')
gguf_files = list(gguf_dir.glob('*.gguf')) if gguf_dir.exists() else []

if not gguf_files:
    print('No GGUF file found — skipping Ollama setup.')
    print('Run Cell 7 first, or export manually.')
else:
    gguf_path = gguf_files[0]
    print(f'GGUF file: {gguf_path.name} ({gguf_path.stat().st_size/1024/1024:.0f}MB)')

    # Create Modelfile
    modelfile = f'''FROM {gguf_path.resolve()}
SYSTEM "You are Jemma SafeBrain, a civic AI assistant for the Town of Normal, Illinois, Illinois State University, and Chicago civic services. You provide accurate information from public records, prioritize safety, and refuse harmful requests."
PARAMETER temperature 0.7
PARAMETER num_ctx 4096
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<eos>"
'''
    modelfile_path = gguf_dir / 'Modelfile'
    modelfile_path.write_text(modelfile)
    print(f'Modelfile written to {modelfile_path}')

    # Check Ollama
    try:
        resp = urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3)
        print('\u2705 Ollama is running!')
        print(f'\nTo register the model, run in terminal:')
        print(f'  ollama create jemma-safebrain -f "{modelfile_path.resolve()}"')
        print(f'\nThen test with:')
        print(f'  ollama run jemma-safebrain "How do I report a pothole in Chicago?"')
    except Exception:
        print('\u26a0\ufe0f Ollama is not running.')
        print('Install from: https://ollama.com/download')
        print(f'Then: ollama create jemma-safebrain -f "{modelfile_path.resolve()}"')

In [ ]:
# Cell 9: Push to HuggingFace Hub (Optional)
PUSH_TO_HUB = False  # Set to True to push
HF_REPO = 'soumitty/jemma-safebrain-gemma-4-e4b-it'

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # will prompt for token

    model.push_to_hub(HF_REPO, private=False)
    tokenizer.push_to_hub(HF_REPO, private=False)
    print(f'\u2705 Pushed to https://huggingface.co/{HF_REPO}')
else:
    print('Set PUSH_TO_HUB = True to upload to HuggingFace')
    print(f'Target repo: {HF_REPO}')

## 🎉 Done!

### What just happened:
1. Detected your GPU and picked optimal settings
2. Loaded 8,000+ civic training samples (Chicago 311, crimes, food inspections, traffic, libraries, safety)
3. Fine-tuned Gemma 4 with QLoRA via Unsloth (2x faster, 60% less memory)
4. Tested the model on civic questions + safety refusals
5. Exported to GGUF for Ollama deployment

### Next steps:
- Run more training iterations with `pipeline/overnight_trainer.py`
- Evaluate with `lm-eval-harness` for proper benchmarks
- Deploy via Ollama: `ollama create jemma-safebrain -f output/gguf/Modelfile`
- Push to HuggingFace: Set `PUSH_TO_HUB = True` in Cell 9